<a href="https://colab.research.google.com/github/ramailh02/UTS-BIG-DATA_RAMA/blob/main/UTS_BIG%20DATA_RAMA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install google-play-scraper

In [ ]:
from google_play_scraper import reviews, Sort
import csv

result, _ = reviews(
    'com.shopee.id',
    lang='id',
    country='id',
    sort=Sort.NEWEST,
    count=100,
    filter_score_with=None
)

filename = 'ulasan_google_play.csv'


with open(filename, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['userName', 'score', 'at', 'content'])
    writer.writeheader()
    for review in result:

        writer.writerow({
            'userName': review['userName'],
            'score': review['score'],
            'at': review['at'],
            'content': review['content']
        })

print(f"Berhasil menyimpan {len(result)} ulasan ke '{filename}'")

In [ ]:
import pandas as pd
from transformers import pipeline

# Load the CSV file into a pandas DataFrame
df_reviews = pd.read_csv(filename)

print("Reviews loaded into DataFrame:")
display(df_reviews.head())

Now, let's load the sentiment analysis model `w11wo/indonesian-roberta-base-prdect-id`.

In [ ]:
sentiment_analyzer = pipeline("sentiment-analysis", model="w11wo/indonesian-roberta-base-prdect-id")

# Perform sentiment analysis on the 'content' column
def get_sentiment(text):
    if pd.isna(text):
        return None, None
    result = sentiment_analyzer(text)[0]
    return result['label'], result['score']

df_reviews[['sentiment_label', 'sentiment_score']] = df_reviews['content'].apply(lambda x: pd.Series(get_sentiment(x)))

print("Sentiment analysis complete. Displaying reviews with sentiments:")
display(df_reviews[['userName', 'content', 'sentiment_label', 'sentiment_score']].head())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set the aesthetic style of the plots
sns.set_style("whitegrid")

# Create a figure with two subplots
plt.figure(figsize=(14, 6))

# Subplot 1: Distribution of Sentiment Labels
plt.subplot(1, 2, 1) # 1 row, 2 columns, first plot
sns.countplot(x='sentiment_label', data=df_reviews, hue='sentiment_label', palette='viridis', legend=False)
plt.title('Distribution of Sentiment Labels')
plt.xlabel('Sentiment Label')
plt.ylabel('Number of Reviews')

# Subplot 2: Distribution of Sentiment Scores
plt.subplot(1, 2, 2) # 1 row, 2 columns, second plot
sns.histplot(df_reviews['sentiment_score'], bins=20, kde=True, color='skyblue')
plt.title('Distribution of Sentiment Scores')
plt.xlabel('Sentiment Score')
plt.ylabel('Number of Reviews')

plt.tight_layout() # Adjust layout to prevent overlapping titles/labels
plt.show()